## 1. Import

`src` klasörü `sys.path`'e eklenir, `silver_transforms` modülü import edilir.

Çıktı: hata yoksa import tamam. `.py` dosyası değişirse bu hücre tekrar çalıştırılır.

In [0]:
import os
import sys
import importlib
from pyspark.sql import functions as F

notebook_dir = os.getcwd()
repo_root = os.path.dirname(notebook_dir)
src_path = os.path.join(repo_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
importlib.reload(silver_transforms)

print(src_path)
print(silver_transforms.CDC_TABLES)

## 2. Landing batch'ini okuma

`read_batch(table_name, batch_id)`: landing'deki bir batch'i explicit schema ile okur.

Veri kolonları bronze tablodan alınır (metadata kolonları hariç, hepsi string). `batch_id` int, `change_ts` önce string okunur, sonra timestamp'e çevrilir.

Test: card batch 1. Çıktıda schema, `op_type` sayıları (beklenen I 2, U 7, D 4) ve `null_change_ts` = 0 görünmeli.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

LANDING_PATH = "/Volumes/workspace/bronze/landing"


def read_batch(table_name, batch_id):
    bronze_columns = spark.table(f"workspace.bronze.{table_name}").columns

    fields = []
    for column_name in bronze_columns:
        if column_name.startswith("_"):
            continue
        fields.append(StructField(column_name, StringType()))
    fields.append(StructField("op_type", StringType()))
    fields.append(StructField("change_ts", StringType()))
    fields.append(StructField("batch_id", IntegerType()))
    batch_schema = StructType(fields)

    batch_folder = f"batch_{batch_id:04d}"
    batch_path = f"{LANDING_PATH}/{table_name}/{batch_folder}/"

    df_raw = spark.read.schema(batch_schema).json(batch_path)
    df_batch = df_raw.withColumn("change_ts", F.to_timestamp("change_ts"))
    return df_batch


df_card_batch = read_batch("card", 1)

df_card_batch.printSchema()
df_card_batch.groupBy("op_type").count().orderBy("op_type").show()
df_card_batch.select(F.count_if(F.col("change_ts").isNull()).alias("null_change_ts")).show()

## 3. Batch içi dedup

Bir key aynı batch'te birden fazla kez gelebilir (batch 2: U → U, I → U). MERGE'te source'ta key başına tek satır olmalı; yoksa MERGE hata verir.

Kural 1: Key başına en son `change_ts`'li kayıt kalır (`ROW_NUMBER`).

Kural 2: Key'in batch'te I kaydı varsa ve son kaydı D değilse son kaydın `op_type`'ı I yapılır (I → U kartı silver'da yok, son haliyle insert edilmeli).

Çıktı: `df_latest_card`; satır sayısı = farklı key sayısı, `op_type` sayıları.

In [0]:
from pyspark.sql import Window

df_card_batch = read_batch("card", 1)
#df_card_batch.filter(F.col("card_id") == "1253").show()

df_card_batch_step01 = df_card_batch.withColumn("rn", 
                                                F.row_number().over(Window.partitionBy("card_id").orderBy(F.desc("change_ts")))# SQL: ROW_NUMBER() OVER (PARTITION BY card_id ORDER BY change_ts DESC)
                                               ) 
df_card_batch_step01.filter(F.col("card_id") == "1253").show()

is_insert=F.when(F.col("op_type") =="I", F.lit(1)).otherwise(F.lit(0))
has_insert=F.max(is_insert).over(Window.partitionBy("card_id")) # SQL: MAX(CASE WHEN op_type = 'I' THEN 1 ELSE 0 END) OVER (PARTITION BY card_id)
df_card_batch_step02 = df_card_batch_step01.withColumn("has_insert", has_insert)
#df_card_batch_step02.filter(F.col("card_id") == "1253").show()

df_card_batch_step02_rn=df_card_batch_step02.filter(F.col("rn") == 1)
#df_card_batch_step02_rn.filter(F.col("card_id") == "1253").show()
df_card_batch_step03 = df_card_batch_step02_rn.withColumn("op_type", F.when((F.col("has_insert")==1) & (F.col("op_type") !="D"),"I").otherwise(F.col("op_type")))
#df_card_batch_step03.filter(F.col("card_id") == "1253").show()
df_card_batch_step04=df_card_batch_step03.drop("rn", "has_insert")

df_card_dedup = df_card_batch_step04
df_card_dedup.filter(F.col("card_id") == "1253").show()
print("Distinct_keys: ",df_card_dedup.select("card_id").distinct().count())
print("Total_keys: ",df_card_dedup.count())
df_card_dedup.groupBy("op_type").count().show()


## 4. Tip dönüşümü ve MERGE source'u

`df_card_dedup` `to_silver` ile silver tiplerine çevrilir. `to_silver` CDC kolonlarını (`op_type`, `change_ts`, `batch_id`) attığı için bu kolonlar key üzerinden geri join edilir; dedup'tan sonra key başına tek satır olduğu için join güvenli.

Sonuç: `df_card_source` → `source_card` temp view'ı (MERGE'ün `USING` kısmı).

Kontrol: source key'leri target'ta var mı? Batch 1 için beklenen: D 4 → true, U 7 → true, I 2 → false. Target `card` MERGE öncesi 892 satır.

In [0]:
table_name = "card"
key_column = "card_id"

df_card_data = silver_transforms.to_silver(df_card_dedup, table_name)
df_card_cdc = df_card_dedup.select(
    F.col(key_column).cast("int").alias(key_column),
    "op_type",
    "change_ts",
    "batch_id",
)

df_card_source = df_card_data.join(df_card_cdc, on=key_column, how="inner")
df_card_source.createOrReplaceTempView("w_source_card")

df_card_source.printSchema()
print("source rows:", df_card_source.count())

# MERGE öncesi kontrol: hangi source key'leri target'ta var
df_card_target = spark.table("workspace.silver.card")
print("target rows (before):", df_card_target.count())

df_target_keys = df_card_target.select(key_column).withColumn("in_target", F.lit(True))
df_check = df_card_source.join(df_target_keys, on=key_column, how="left")
df_check = df_check.withColumn("in_target", F.coalesce(F.col("in_target"), F.lit(False)))
df_check.groupBy("op_type", "in_target").count().orderBy("op_type").show()

## 5. MERGE INTO (card, batch 1)

Target: `workspace.silver.card` (alias `t`). Source: `source_card` view'ı (alias `s`). Eşleşme: `card_id`.

| Key target'ta | op_type | Ek koşul | İşlem |
|---|---|---|---|
| var | D | – | DELETE (hard delete; CDF'te `delete`) |
| var | U | `s.change_ts` > `t._change_ts` | UPDATE; `_change_ts` = `s.change_ts` |
| yok | I | – | INSERT; `_change_ts` = `s.change_ts` |

Beklenen sonuç (batch 1): `num_deleted_rows` 4, `num_updated_rows` 7, `num_inserted_rows` 2; target 892 + 2 − 4 = **890 satır**; batch'teki 7 U kartının hepsi `gold`.

In [0]:
%sql
MERGE INTO workspace.silver.card AS t
USING w_source_card AS s
ON t.card_id = s.card_id

-- Kart iptali: key target'ta var ve op_type D ise satır silinir (hard delete)
WHEN MATCHED AND s.op_type = 'D' THEN
  DELETE

-- Güncelleme: key var, op_type U ve gelen kayıt target'tan yeniyse update edilir
WHEN MATCHED AND s.op_type = 'U' AND s.change_ts > t._change_ts THEN
  UPDATE SET
    t.disp_id = s.disp_id,
    t.type = s.type,
    t.issued_date = s.issued_date,
    t._change_ts = s.change_ts

-- Yeni kart: key target'ta yok ve op_type I ise insert edilir
WHEN NOT MATCHED AND s.op_type = 'I' THEN
  INSERT (card_id, disp_id, type, issued_date, _change_ts)
  VALUES (s.card_id, s.disp_id, s.type, s.issued_date, s.change_ts);

In [0]:
%sql
SELECT COUNT(*) AS target_rows FROM workspace.silver.card

## 6. Tüm tablolar ve batch'ler: `prepare_source`

`prepare_source(table_name, batch_id)`: hücre 4–8'deki adımların tek function hali. Batch'i okur, dedup yapar, silver tiplerine çevirir ve `vw_source_<tablo>` temp view'ını (MERGE source'u) oluşturur.

`BATCH_ID` ile batch seçilir; döngü 4 tablonun view'ını hazırlar. Sonra hücre 15, 17, 19, 21 (MERGE) ve 22 (COUNT) çalıştırılır. Batch'ler sırayla: 1 → 2 → 3.

Çıktı: tablo başına source satır sayısı = farklı key sayısı ve `op_type` dağılımı.

| Batch | client | account | card | loan |
|---|---|---|---|---|
| 1 | 13 (I 5, U 8) | 12 (U 12) | 13 (I 2, U 7, D 4) | 8 (U 8) |
| 2 | 24 (I 5, U 19) | 19 (U 19) | 15 (I 5, U 8, D 2) | 6 (U 6) |
| 3 | 9 (I 4, U 5) | 5 (U 5) | 12 (I 1, U 6, D 5) | 11 (U 11) |

Batch 2'de client 26 → 24 (435 ve 4206 dedup), card 16 → 15 (1253 I → U birleşti, I kaldı).

In [0]:
from pyspark.sql import Window

KEY_COLUMNS = {
    "client": "client_id",
    "account": "account_id",
    "card": "card_id",
    "loan": "loan_id",
}


def prepare_source(table_name, batch_id):
    key_column = KEY_COLUMNS[table_name]
    df_batch = read_batch(table_name, batch_id)

    # Dedup kural 1: key başına en yeni change_ts 1 numara alır
    window_latest = Window.partitionBy(key_column).orderBy(F.col("change_ts").desc())
    df_ranked = df_batch.withColumn("rn", F.row_number().over(window_latest))

    # Dedup kural 2 hazırlığı: key'in batch'te I kaydı var mı (1 = var)
    window_key = Window.partitionBy(key_column)
    is_insert = F.when(F.col("op_type") == "I", 1).otherwise(0)
    df_ranked = df_ranked.withColumn("has_insert", F.max(is_insert).over(window_key))

    df_dedup = df_ranked.filter(F.col("rn") == 1)

    # Dedup kural 2: I → U ise kayıt target'ta yok, son haliyle insert edilmeli
    final_op_type = (
        F.when((F.col("has_insert") == 1) & (F.col("op_type") != "D"), "I")
        .otherwise(F.col("op_type"))
    )
    df_dedup = df_dedup.withColumn("op_type", final_op_type)
    df_dedup = df_dedup.drop("rn", "has_insert")

    # to_silver CDC kolonlarını atar; bu kolonlar ayrı tutulur
    df_data = silver_transforms.to_silver(df_dedup, table_name)
    df_cdc = df_dedup.select(
        F.col(key_column).cast("int").alias(key_column),
        "op_type",
        "change_ts",
        "batch_id",
    )

    # Join güvenli: dedup sonrası key başına tek satır
    df_source = df_data.join(df_cdc, on=key_column, how="inner")

    # SOURCE: MERGE'ün USING kısmında kullanılacak view
    view_name = f"vw_source_{table_name}"
    df_source.createOrReplaceTempView(view_name)
    return df_source


BATCH_ID = 3

# Döngü: aynı batch için 4 tablonun source view'ı hazırlanır
for table_name in silver_transforms.CDC_TABLES:
    key_column = KEY_COLUMNS[table_name]
    df_source = prepare_source(table_name, BATCH_ID)

    source_rows = df_source.count()
    source_keys = df_source.select(key_column).distinct().count()
    print(f"batch {BATCH_ID} | vw_source_{table_name}: rows={source_rows}, keys={source_keys}")
    df_source.groupBy("op_type").count().orderBy("op_type").show()

## 7. MERGE: card (I / U / D)

Source: `vw_source_card` (hücre 13, seçili `BATCH_ID`). Target: `workspace.silver.card`.

D → delete, U → yeni `change_ts` ise update, I → insert.

| Batch | deleted | updated | inserted | Target satır |
|---|---|---|---|---|
| 1 | 0 | 0 | 0 | 890 (batch 1 hücre 10'da uygulandı; tekrar = 0/0/0) |
| 2 | 2 | 8 | 5 | 893 |
| 3 | 4 | 6 | 1 | 890 |

Batch 3'te D 5 geliyor ama 4 silinir: `card_id` 101254 target'ta yok, sessizce atlanır.

In [0]:
%sql
-- SOURCE: vw_source_card (seçili batch) | TARGET: workspace.silver.card
MERGE INTO workspace.silver.card AS t
USING vw_source_card AS s
ON t.card_id = s.card_id

-- Kart iptali: key target'ta var ve op_type D ise satır silinir (hard delete)
WHEN MATCHED AND s.op_type = 'D' THEN
  DELETE

-- Güncelleme: key var, op_type U ve gelen kayıt target'tan yeniyse update edilir
WHEN MATCHED AND s.op_type = 'U' AND s.change_ts > t._change_ts THEN
  UPDATE SET
    t.disp_id = s.disp_id,
    t.type = s.type,
    t.issued_date = s.issued_date,
    t._change_ts = s.change_ts

-- Yeni kart: key target'ta yok ve op_type I ise insert edilir
WHEN NOT MATCHED AND s.op_type = 'I' THEN
  INSERT (card_id, disp_id, type, issued_date, _change_ts)
  VALUES (s.card_id, s.disp_id, s.type, s.issued_date, s.change_ts)

## 8. MERGE: client (I / U)

Source: `vw_source_client` (hücre 13, seçili `BATCH_ID`). Target: `workspace.silver.client`.

U → yeni `change_ts` ise update (adres değişimi, `district_id`), I → insert. `client`'ta D yok.

| Batch | updated | inserted | Target satır |
|---|---|---|---|
| 1 | 8 | 5 | 5.369 → 5.374 |
| 2 | 19 | 5 | 5.379 |
| 3 | 5 | 4 | 5.383 |

Batch 2'de 435 ve 4206 dedup ile tek satıra indi; son `district_id`'leri 28 ve 27 olmalı.

In [0]:
%sql
-- SOURCE: vw_source_client (seçili batch) | TARGET: workspace.silver.client
MERGE INTO workspace.silver.client AS t
USING vw_source_client AS s
ON t.client_id = s.client_id

-- Adres değişimi: key var, op_type U ve gelen kayıt target'tan yeniyse update edilir
WHEN MATCHED AND s.op_type = 'U' AND s.change_ts > t._change_ts THEN
  UPDATE SET
    t.birth_number = s.birth_number,
    t.district_id = s.district_id,
    t._change_ts = s.change_ts

-- Yeni müşteri: key target'ta yok ve op_type I ise insert edilir
WHEN NOT MATCHED AND s.op_type = 'I' THEN
  INSERT (client_id, birth_number, district_id, _change_ts)
  VALUES (s.client_id, s.birth_number, s.district_id, s.change_ts)

## 9. MERGE: account (U)

Source: `vw_source_account` (hücre 13, seçili `BATCH_ID`). Target: `workspace.silver.account`.

Sadece U: yeni `change_ts` ise update (`frequency` değişimi). `account`'ta I ve D yok; satır sayısı 4.500'de kalır.

| Batch | updated | Target satır |
|---|---|---|
| 1 | 12 | 4.500 |
| 2 | 19 | 4.500 |
| 3 | 5 | 4.500 |

In [0]:
%sql
-- SOURCE: vw_source_account (seçili batch) | TARGET: workspace.silver.account
MERGE INTO workspace.silver.account AS t
USING vw_source_account AS s
ON t.account_id = s.account_id

-- Frequency değişimi: key var, op_type U ve gelen kayıt target'tan yeniyse update edilir
WHEN MATCHED AND s.op_type = 'U' AND s.change_ts > t._change_ts THEN
  UPDATE SET
    t.district_id = s.district_id,
    t.frequency = s.frequency,
    t.open_date = s.open_date,
    t._change_ts = s.change_ts

## 10. MERGE: loan (U)

Source: `vw_source_loan` (hücre 13, seçili `BATCH_ID`). Target: `workspace.silver.loan`.

Sadece U: yeni `change_ts` ise update (`status` C → A / D). `loan`'da I ve D yok; satır sayısı 682'de kalır.

| Batch | updated | Target satır |
|---|---|---|
| 1 | 8 | 682 |
| 2 | 6 | 682 |
| 3 | 10 | 682 |

Batch 3'te U 11 geliyor ama 10 update olur: `loan_id` 107308 target'ta yok, sessizce atlanır.

In [0]:
%sql
-- SOURCE: vw_source_loan (seçili batch) | TARGET: workspace.silver.loan
MERGE INTO workspace.silver.loan AS t
USING vw_source_loan AS s
ON t.loan_id = s.loan_id

-- Kredi durumu değişimi: key var, op_type U ve gelen kayıt target'tan yeniyse update edilir
WHEN MATCHED AND s.op_type = 'U' AND s.change_ts > t._change_ts THEN
  UPDATE SET
    t.account_id = s.account_id,
    t.loan_date = s.loan_date,
    t.amount = s.amount,
    t.duration = s.duration,
    t.payments = s.payments,
    t.status = s.status,
    t._change_ts = s.change_ts

In [0]:
%sql
-- TARGET özeti: her batch'in MERGE'lerinden sonra çalıştırılır
-- changed_rows: _change_ts başlangıç değerinden (1998-12-31) büyük olan, yani en az bir kez MERGE ile yazılmış satırlar
SELECT 'client' AS table_name,
       COUNT(*) AS target_rows,
       COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00') AS changed_rows,
       MAX(_change_ts) AS max_change_ts
FROM workspace.silver.client
UNION ALL
SELECT 'account', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.account
UNION ALL
SELECT 'card', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.card
UNION ALL
SELECT 'loan', COUNT(*), COUNT_IF(_change_ts > TIMESTAMP '1998-12-31 00:00:00'), MAX(_change_ts)
FROM workspace.silver.loan